In [1]:
#imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
import re
import string
from collections import Counter
import gc
import numpy as np
import torch.nn.functional as F

#data shapes
imdb = load_dataset("imdb")
sst2 = {
    "train": load_dataset("SetFit/sst2", split="train"),
    "test": load_dataset("SetFit/sst2", split="test")
}
# === Preprocessing ===
def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# === Tokenization & Vocab ===
def tokenize(text):
    return clean_text(text).split()

# Build vocab from IMDB
counter = Counter()
for example in imdb["train"]:
    counter.update(tokenize(example["text"]))

# Only keep words with min_freq ≥ 5
min_freq = 5
vocab = {"<pad>": 0, "<unk>": 1}
for word, freq in counter.items():
    if freq >= min_freq:
        vocab[word] = len(vocab)
inv_vocab = {v: k for k, v in vocab.items()}
PAD_IDX = vocab["<pad>"]
UNK_IDX = vocab["<unk>"]

def text_pipeline(text, max_len=256):
    tokens = tokenize(text)
    ids = [vocab.get(token, UNK_IDX) for token in tokens[:max_len]]
    return torch.tensor(ids, dtype=torch.long)

# === Dataset Wrappers ===
class TextDataset(Dataset):
    def __init__(self, hf_dataset, label_transform, max_len=256):
        self.data = hf_dataset
        self.label_transform = label_transform
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text_field = item["text"] if "text" in item else item["content"]
        tokens = text_pipeline(text_field, max_len=self.max_len)
        label = torch.tensor(self.label_transform(item["label"]), dtype=torch.long)
        return tokens, label

def collate_batch(batch):
    texts, labels = zip(*batch)
    padded = pad_sequence(texts, batch_first=True, padding_value=PAD_IDX)
    return padded, torch.stack(labels)

imdb_dataset = TextDataset(imdb["train"], lambda x: x)
sst2_dataset = TextDataset(sst2["train"], lambda x: x)

imdb_loader = DataLoader(imdb_dataset, batch_size=64, shuffle=True, collate_fn=collate_batch)
sst2_loader = DataLoader(sst2_dataset, batch_size=256, shuffle=True, collate_fn=collate_batch)

class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TextClassifier(len(vocab), num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


#sliced wasserstein
from DP_ACE_Materials.sliced_wasserstein import sliced_wasserstein_distance

def feature_extractor(loader, max_samples=1000):
    features = []
    model.eval()
    sample_count = 0
    
    with torch.no_grad():
        for x, _ in loader:
            if sample_count >= max_samples:
                break
            
            feature = model(x)
            features.append(feature.numpy())
            sample_count += x.shape[0]
        return np.concatenate(features, axis=0)
    
imdb_features = feature_extractor(imdb_loader)
print("Extracted imdb features")
 
sst2_features = feature_extractor(sst2_loader)
print("Extracted amazon features")

print("Sliced Wasserstein Distance between amazon and imdb:", 
      sliced_wasserstein_distance(sst2_features, imdb_features))

wass = sliced_wasserstein_distance(sst2_features, imdb_features)

Repo card metadata block was not found. Setting CardData to empty.
Repo card metadata block was not found. Setting CardData to empty.


Extracted imdb features
Extracted amazon features
Sliced Wasserstein Distance between amazon and imdb: 0.017000496109202137


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
import re
import string
from collections import Counter
import gc
import numpy as np
from opacus import PrivacyEngine
from opacus.layers import DPLSTM
from privacy_analysis.compute_privacy_sgm import compute_dp_sgd_privacy
from DP_ACE_Materials.Wasserstein_DP_ACE import DP_ACE

In [3]:
imdb = load_dataset("imdb")
sst2 = load_dataset("SetFit/sst2")

max_imdb_contribution = len(imdb["train"])
max_sst2_contribution = len(sst2["train"])

total_if_imdb_maxed = int(len(imdb["train"]) / 0.96)
total_if_sst2_maxed = int(len(sst2["train"]) / 0.04)
total_desired_size = min(total_if_imdb_maxed, total_if_sst2_maxed)
imdb_size = int(0.96 * total_desired_size)
sst2_size = int(0.04 * total_desired_size)

np.random.seed(42)
imdb_indices = np.random.choice(len(imdb["train"]), size=len(imdb), replace=False)
sst2_indices = np.random.choice(len(sst2["train"]), size=6250, replace=False)

imdb_subset = imdb["train"]
sst2_subset = sst2["train"].select(sst2_indices)

Repo card metadata block was not found. Setting CardData to empty.
08/26/2025 21:11:37:WARNING:Repo card metadata block was not found. Setting CardData to empty.


In [4]:
def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    return clean_text(text).split()

counter = Counter()
for example in imdb["train"]:
    counter.update(tokenize(example["text"]))

min_freq = 3
vocab = {"<pad>": 0, "<unk>": 1}
for word, freq in counter.items():
    if freq >= min_freq:
        vocab[word] = len(vocab)
inv_vocab = {v: k for k, v in vocab.items()}
PAD_IDX = vocab["<pad>"]
UNK_IDX = vocab["<unk>"]

def text_pipeline(text, max_len=256):
    tokens = tokenize(text)[:max_len]
    bow_vector = torch.zeros(len(vocab), dtype=torch.float)
    for token in tokens:
        idx = vocab.get(token, UNK_IDX)
        bow_vector[idx] += 1
    return bow_vector

In [5]:
class TextDataset(Dataset):
    def __init__(self, hf_dataset, label_transform, text_field, max_len=256):
        self.data = hf_dataset
        self.label_transform = label_transform
        self.max_len = max_len
        self.text_field = text_field

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = text_pipeline(item[self.text_field], max_len=self.max_len)
        label = torch.tensor(self.label_transform(item["label"]), dtype=torch.long)
        return tokens, label

def collate_batch(batch):
    texts, labels = zip(*batch)
    padded = pad_sequence(texts, batch_first=True, padding_value=PAD_IDX)
    return padded, torch.stack(labels)

In [6]:
train_loader_public = DataLoader(
    TextDataset(sst2_subset, label_transform=lambda x: x, text_field="text"),
    batch_size=64, shuffle=True, collate_fn=collate_batch
)

train_loader_private = DataLoader(
    TextDataset(imdb_subset, label_transform=lambda x: x, text_field="text"),
    batch_size=64, shuffle=True, collate_fn=collate_batch
)

test_loader = DataLoader(
    TextDataset(imdb["test"], label_transform=lambda x: x, text_field="text"),
    batch_size=64, shuffle=False, collate_fn=collate_batch
)

In [7]:
class Model(nn.Module):
    def __init__(self, vocab_size, num_classes=2):
        super().__init__()
        self.fc = nn.Linear(vocab_size, num_classes)

    def forward(self, x):
        return self.fc(x)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model(len(vocab), num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
privacy_engine = PrivacyEngine()
gc.collect()

/opt/anaconda3/lib/python3.12/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


0

In [9]:
train_losses, train_accuracies, test_losses, test_accuracies = DP_ACE(
                                                                        model=model,
                                                                        D_loader=train_loader_private,
                                                                        D_s_loader=train_loader_public,
                                                                        test_loader=test_loader,
                                                                        lr=0.01, 
                                                                        sigma=0.716, 
                                                                        l=criterion,
                                                                        T=15,
                                                                        delta=1e-5,
                                                                        wass = wass
                                                                      )

DP-SGD with sampling rate = 0.256% and noise_multiplier = 0.716 iterated over 391 steps satisfies differential privacy with eps = 1.95 and delta = 1e-05.
The optimal RDP order is 6.0.
Epoch [1/15] ... Training Loss: 0.6933 ... Training Accuracy: 49.05% ... Test Loss: 0.6918 ... Test Accuracy: 52.08% ... Epsilon: 1.9501 ... Delta: 1e-05
DP-SGD with sampling rate = 0.256% and noise_multiplier = 0.716 iterated over 782 steps satisfies differential privacy with eps = 2.12 and delta = 1e-05.
The optimal RDP order is 5.5.
Epoch [2/15] ... Training Loss: 0.6922 ... Training Accuracy: 56.70% ... Test Loss: 0.6834 ... Test Accuracy: 58.34% ... Epsilon: 2.1208 ... Delta: 1e-05
DP-SGD with sampling rate = 0.256% and noise_multiplier = 0.716 iterated over 1172 steps satisfies differential privacy with eps = 2.19 and delta = 1e-05.
The optimal RDP order is 5.5.
Epoch [3/15] ... Training Loss: 0.6914 ... Training Accuracy: 60.37% ... Test Loss: 0.6765 ... Test Accuracy: 60.68% ... Epsilon: 2.1915 ..